# Generative AI APIs in Python
### ISA 383: Python for Business  |  American University of Sharjah

**Learning Objectives**

- Understand how Generative AI APIs fit into the same request/response mental model you already saw in Lecture 14.
- Create a personal **Google AI Studio** key and make your first call with the `google-genai` SDK.
- Build a small **chatbot** that holds a conversation, give it a persona, and tune it with temperature and max tokens.
- Understand the concept of **Retrieval-Augmented Generation (RAG)** and why it is needed.
- Build a complete RAG pipeline end-to-end using Gemini's embedding model and pure NumPy -- no vector databases, no downloads.

**Philosophy of this lab**

Last week we saw that almost every data API in Python is really just `requests.get(...)` with a friendly wrapper. Generative AI APIs are no different: you send a prompt, the server sends back generated text. What changes is *what* the server does in between -- and that is where things get interesting.

---

## Using this notebook in Google Colab

1. Before editing, select **File > Save a copy in Drive**.
2. Run the cells in order. Some practice cells intentionally wait for your input.
3. The notebook creates any course folders and teaching files it needs automatically.
4. Files under `/content` are temporary and disappear when the Colab runtime resets.
5. Never paste an API key into a notebook cell. Use the **Secrets** panel when instructed.

You do not need to find, copy, or type a file path for the prepared course data.


## 1. Why Gemini, and Why Now?

For a classroom lab we need an LLM API that ticks four boxes:

1. **A classroom-accessible tier, subject to current regional availability and quotas.**
2. **Cloud-hosted** -- students on a laptop with 8 GB of RAM should not have to download a 10-GB model.
3. **Chat generation and embeddings under one key** (so we can build RAG without juggling two accounts).
4. **A mature Python SDK** that hides the raw HTTP calls.

Google's **Gemini** API meets these requirements, so it is the service used in this course. Other providers may offer similar capabilities, but their pricing, quotas, and model access change over time.

**Regional caveat.** API availability and free-tier eligibility depend on the account and region. If you receive a location or billing error, check the current [Gemini API pricing documentation](https://ai.google.dev/gemini-api/docs/pricing).

## 2. Getting Your Own API Key

1. Go to <https://aistudio.google.com/api-keys>.
2. Sign in with a Google account (any Gmail works).
3. Click **"Create API key"**.
4. Copy the key -- it is a long string starting with `AIza...`.

**Treat it like a password.** Never paste it into a screenshot or a slide; never commit it to GitHub. We will load it from an environment variable the same way you loaded the NASA key last week.

## 3. Install and import

Run the next cell once. Colab installs the current `google-genai` SDK automatically.


In [ ]:
%pip install -q google-genai

print("Google GenAI SDK is ready.")


In [1]:
import os
from pathlib import Path

from google import genai
from google.genai import types

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import textwrap, time, warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (9, 4.5), "font.size": 10})

print("google-genai version:", genai.__version__)

google-genai version: 1.73.1


### 3.1 Load your key from Colab Secrets

1. Open the **Secrets** panel using the key icon on the left.
2. Create a secret named `GEMINI_API_KEY`.
3. Allow this notebook to access that secret.

The secret belongs to your Google account and is not saved inside the notebook.


In [ ]:
from google.colab import userdata

try:
    key = userdata.get("GEMINI_API_KEY")
except Exception as error:
    raise RuntimeError(
        "Add GEMINI_API_KEY in the Colab Secrets panel and allow notebook access."
    ) from error

print("Gemini API key loaded securely.")


In [3]:
client = genai.Client(api_key=key)
print("Client ready.")

Client ready.


## 4. Your First Generative Call

Every call to Gemini goes through `client.models.generate_content(...)`. It takes two things you care about:

- `model`: which LLM to use (e.g. `"gemini-2.5-flash"`).
- `contents`: the prompt -- a string or a list of messages.

The response object has a `.text` attribute with the model's reply, plus metadata like token usage.

In [4]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="In one sentence, explain what an API is to a business student.",
)
print(response.text)

An API (Application Programming Interface) is a software connector that allows different applications to share data and functionality, enabling businesses to integrate systems, automate processes, and build new services.


In [5]:
# Peek at the metadata Gemini returns
print("Model version :", response.model_version)
print("Finish reason :", response.candidates[0].finish_reason)
print("Input tokens  :", response.usage_metadata.prompt_token_count)
print("Output tokens :", response.usage_metadata.candidates_token_count)
print("Total tokens  :", response.usage_metadata.total_token_count)

Model version : gemini-2.5-flash
Finish reason : FinishReason.STOP
Input tokens  : 15
Output tokens : 35
Total tokens  : 1271


That is the entire pattern:

```
client.models.generate_content(model=..., contents=...)  ->  response  ->  response.text
```

Everything else in this notebook is just putting this call into more interesting shapes.

## 5. Customising the Model

Three things you will tweak on almost every call:

| Option | What it does | Typical value |
|---|---|---|
| `model`              | Which model to use                           | `"gemini-2.5-flash"` (fast) or `"gemini-2.5-pro"` (smarter) |
| `system_instruction` | A persona / set of rules baked into the prompt | a short paragraph                         |
| `temperature`        | 0 = deterministic; 1 = creative              | 0.2 for factual, 0.9 for brainstorming    |
| `max_output_tokens`  | Upper bound on reply length                  | 200 -- 2000                                |

Pass them via a `GenerateContentConfig` object.

In [6]:
def ask(prompt, *, model="gemini-2.5-flash", system_instruction=None,
        temperature=0.7, max_output_tokens=4000):
    """Single-turn helper: send a prompt, return the text reply."""
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        max_output_tokens=max_output_tokens,
    )
    resp = client.models.generate_content(model=model, contents=prompt, config=config)
    return resp.text


# A default call
ask("Give me three bullet points on why UAE introduced VAT in 2018.")

"The UAE introduced Value Added Tax (VAT) in 2018 primarily for the following reasons:\n\n*   **Diversification of Government Revenue:** To reduce the country's reliance on oil and gas revenues, providing a more stable and sustainable source of income for the government, especially given fluctuations in global oil prices.\n*   **Funding Public Services and Infrastructure:** To generate additional funds for government spending on essential public services such as healthcare, education, and the development of world-class infrastructure, supporting the UAE's ambitious growth and development plans.\n*   **Enhancing Fiscal Sustainability and Aligning with GCC Agreement:** To strengthen the nation's fiscal stability and sustainability for the long term, and to fulfill its commitment under the GCC (Gulf Cooperation Council) Common VAT Agreement, which aimed for a unified tax framework across member states."

### 5.1 Swapping the model

The only change to switch to a stronger model is the `model=` argument. `gemini-2.5-pro` is slower and has different availability and quota limits that may change, but it reasons noticeably better on hard questions.


In [7]:
question = "A retailer in Sharjah had AED 500,000 in taxable sales last quarter. Estimate the VAT due, and list two edge cases you would check before filing."
fast_answer = ask(question, model="gemini-2.5-flash", temperature=0.2)
print("--- gemini-2.5-flash ---")
print(fast_answer)

--- gemini-2.5-flash ---
Here's an estimation of the VAT due and two edge cases to consider:

---

### Estimated VAT Due

The standard VAT rate in the UAE is **5%**.

Assuming all AED 500,000 in sales are standard-rated (5%) and represent the **output VAT** generated:

*   **Output VAT = Taxable Sales × VAT Rate**
*   Output VAT = AED 500,000 × 0.05
*   **Output VAT = AED 25,000**

**Important Note:** This AED 25,000 represents the *output VAT* collected on sales. The **net VAT due** to the Federal Tax Authority (FTA) will be this output VAT **minus any recoverable input VAT** paid by the retailer on their purchases, expenses, and imports during the same quarter. Without information on input VAT, the maximum VAT due would be AED 25,000.

---

### Two Edge Cases to Check Before Filing

1.  **Mixed Supplies / Zero-Rated / Exempt Sales:**
    *   **Check:** Are *all* AED 500,000 in sales truly standard-rated (5%)? A retailer might have sales that fall under different VAT treatments.
    *

In [8]:
question = "A retailer in Sharjah had AED 500,000 in taxable sales last quarter. Estimate the VAT due, and list two edge cases you would check before filing."
pro_answer = ask(question, model="gemini-2.5-pro", temperature=0.2)
print("--- gemini-2.5-pro ---")
print(pro_answer)

--- gemini-2.5-pro ---
Of course. Here is an estimation of the VAT due and two critical edge cases to consider.

---

### Estimated VAT Due

Based on the information provided, the estimated VAT due is **AED 25,000**.

**Calculation:**

*   **Taxable Sales:** AED 500,000
*   **Standard VAT Rate in the UAE:** 5%
*   **VAT Due (Output Tax):** AED 500,000 * 0.05 = **AED 25,000**

This amount is the *output tax* collected on behalf of the government. However, this is not the final amount the retailer will pay. The final payable amount is calculated after deducting the input tax.

---

### Two Edge Cases to Check Before Filing

Before filing the VAT return and paying the AED 25,000, it is crucial to verify the following two points, as they can significantly change the amount of VAT you actually owe to the Federal Tax Authority (FTA).

#### 1. Recoverable Input VAT

This is the most common and important check. The VAT system allows businesses to reclaim the VAT they paid on their own business

### 5.2 Personas via `system_instruction`

A `system_instruction` is a hidden message that sets the bot's role for every turn. It does not appear in the chat transcript, but the model reads it before every reply.

In [9]:
tutor_persona = textwrap.dedent("""
You are an expert but gentle tutor for a Python-for-Business course at the
American University of Sharjah. You speak in clear, plain English, prefer
short paragraphs, and always anchor your answers in UAE or GCC examples when
relevant. If a student's question is ambiguous, you ask one clarifying
question before answering.
""").strip()

print(ask("What is NumPy and when would I use it?", system_instruction=tutor_persona))

Ahlan! That's a great question, and NumPy is a fundamental tool for anyone working with data in Python.

### What is NumPy?

NumPy, short for "Numerical Python," is the core library for numerical computing in Python. It provides:

*   **Powerful Data Structure:** Its main feature is the `ndarray` (N-dimensional array) object. Think of it as a super-charged list that can hold numbers in one, two, or many dimensions, like a table or a grid.
*   **Speed and Efficiency:** NumPy arrays are much faster and use less memory than standard Python lists when dealing with large amounts of numerical data. This is crucial for performance.
*   **Mathematical Functions:** It comes with a vast collection of high-level mathematical functions to operate on these arrays, making complex calculations very straightforward.

### When Would You Use It?

You'd use NumPy whenever you need to perform fast, efficient numerical operations on collections of data, especially in a business context:

1.  **Data Analysi

### 5.3 Temperature: the "creativity knob"

A lower temperature reduces sampling variability and usually makes answers more consistent, but API responses are not guaranteed to be identical across runs. A higher temperature permits more variation. Try the same prompt twice at `temperature=0` and `temperature=1` and compare the results.

In [10]:
brainstorm = "Invent a short, memorable brand name for a UAE-based fintech startup focused on SME lending. Just give me brand names as a Python list."

print("--- temperature = 0.0 ---")
for i in range(2):
    print(f"[{i+1}]", ask(brainstorm, temperature=0.0, max_output_tokens=500))
print()
print("--- temperature = 1.0 ---")
for i in range(2):
    print(f"[{i+1}]", ask(brainstorm, temperature=1.0, max_output_tokens=500))

--- temperature = 0.0 ---
[1] ```python
[
    "NoorFund",
    "DuneCapital",

[2] ```python
[
    "NoorFund",
    "DuneCapital",


--- temperature = 1.0 ---
[1] ```python
[
    "BarakaFi",
    "TamkeenFi",
    
[2] ```python
brand_names = [
    "NoorFund",
    "Emkan


## 6. Building a Chatbot

`generate_content` is single-turn -- it forgets everything as soon as the call returns. For a real chatbot you want the model to *remember* what was said earlier. The SDK gives us a `Chat` object that manages this state for us.

In [11]:
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=tutor_persona,
        temperature=0.5,
        max_output_tokens=500,
    ),
)

print(chat.send_message("Hi Gemini, you are talking to ISA 383 class where we are learning Python for Business Analytics.").text)

Ahlan wa sahlan, ISA 383 class! It's great to connect with you all.

I'm here to help you navigate Python for your Business Analytics journey. Think of me as your guide for any questions you have about the code, the concepts, or how it all applies to business here in the UAE and GCC.

Please feel free to ask anything. I'm ready when you are!


In [12]:
# The chat object remembers context across calls
print(chat.send_message("Based on what I just told you, recommend one ISA 383 topic I would enjoy most.").text)

That's an interesting challenge! Based *only* on the fact that you're in ISA 383, "Python for Business Analytics," I don't have enough information about your personal interests to pinpoint what *you* would enjoy most.

To give you a good recommendation, could you tell me a little more?

Are you more interested in:

*   **Understanding past performance** (like analyzing sales trends for a Dubai-based retailer)?
*   **Predicting future outcomes** (like forecasting demand for a new product launch in Abu Dhabi)?
*   **Automating tasks and reports** (like streamlining monthly financial summaries for a regional bank)?

Once I know your preference, I can suggest a topic you'd likely find engaging!


In [13]:
# Inspect the full conversation history -- it is a plain list of messages
for turn in chat.get_history():
    text = turn.parts[0].text if turn.parts else ""
    short = text.strip().split("\n")[0][:90]
    print(f"[{turn.role}] {short}...")

[user] Hi Gemini, you are talking to ISA 383 class where we are learning Python for Business Anal...
[model] Ahlan wa sahlan, ISA 383 class! It's great to connect with you all....
[user] Based on what I just told you, recommend one ISA 383 topic I would enjoy most....
[model] That's an interesting challenge! Based *only* on the fact that you're in ISA 383, "Python ...


### 6.1 An interactive loop

Here is a minimal chat REPL you can drop into any notebook. Type `exit` (or interrupt the kernel) to stop.

```python
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(system_instruction=tutor_persona),
)

while True:
    user = input("you> ").strip()
    if user.lower() in {"exit", "quit"}:
        break
    print("bot>", chat.send_message(user).text, "\n")
```

Because `input()` blocks the notebook, we leave this as a code snippet rather than executing it; paste it into a new cell when you want to try it live.

In [14]:
chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(system_instruction=tutor_persona),
)

while True:
    user = input("you> ").strip()
    if user.lower() in {"exit", "quit"}:
        break
    print("bot>", chat.send_message(user).text, "\n")

bot> Hello! Welcome to our Python-for-Business course. I'm ready to help with any questions you have. Just ask away! 



## 7. Where the Base Chatbot Breaks Down

The chatbot we just built is impressive, but it has a hard limit: it only knows what was in its training data. Ask it about our university's internal assessment policy, your employer's HR handbook, or a new regulation issued last week, and it will either hallucinate a plausible-looking answer or politely decline.

Let us see this in action with our fictional company, **Falcon Dunes Trading LLC**.

In [15]:
print(ask("At Falcon Dunes Trading LLC in Dubai, how many days of annual leave do full-time employees get per year?"))

It's highly unlikely that the specific annual leave policy for Falcon Dunes Trading LLC is publicly available. Companies typically keep their detailed HR policies internal.

However, Falcon Dunes Trading LLC, like all companies in Dubai, must comply with the **UAE Labour Law (Federal Decree-Law No. 33 of 2021)**. Under this law, the minimum annual leave entitlement for full-time employees is:

*   **After completing six months but less than one year of service:** 2 days of leave for each month.
*   **After completing one year of service:** 30 calendar days of annual leave per year.

It's important to note that while 30 calendar days is the *minimum* required by law for employees with over a year of service, a company *can* choose to offer more leave days as part of their employment benefits package, but they cannot offer less.


Whatever the model replied above, it did not actually *know* -- Falcon Dunes is a fictional company we invented for this lab. The answer is either a cautious "I do not have this information" or a confident hallucination. Neither is useful if you are trying to build a reliable business assistant.

### 7.1 The idea behind Retrieval-Augmented Generation (RAG)

**RAG** solves this in the most direct way possible: before asking the model a question, we *look up* the relevant documents ourselves and paste them into the prompt. The model then has the answer right there in front of it and does not need to guess.

The pipeline has six stages:

```
 (offline, done once)                 (online, done per question)

   +--------+   +---------+   +----------+      +---------+   +----------+   +-----+
   |  Docs  |-->| Chunker |-->| Embedder |-+    |  Query  |-->| Embedder |-->| Sim |
   +--------+   +---------+   +----------+ |    +---------+   +----------+   +--+--+
                                           v                                    |
                                     +-----------+                              v
                                     |  Vector   |<----------------------- top-k chunks
                                     |   store   |                              |
                                     +-----------+                              v
                                                                          +----------+
                                                                          |   LLM    |--> answer
                                                                          +----------+
```

The first three stages are done once, when you ingest your documents. The last three happen every time a user asks a question. The entire trick is that we never fine-tune the model -- we just hand it the right context at the right moment.

### 7.2 Why not just paste all the documents into the prompt?

A fair question. Three reasons:

1. **Context length.** Gemini 2.5 Flash will take roughly 1 million tokens of input, which is a lot, but still finite -- and enterprise document stores routinely exceed this.
2. **Cost.** Even on the free tier, tokens count against your daily quota. Sending 500 pages for every question is wasteful.
3. **Relevance.** LLMs handle long context better than they used to, but they still pay more attention to *some* of the context than the rest. Giving it the three most relevant paragraphs beats giving it the whole binder.

## 8. Building RAG, Step by Step

We now walk through the full RAG pipeline end-to-end on a small corpus. The notebook prepares six plain-text files automatically in its temporary Colab working folder:

- Three Wikipedia excerpts: `wiki_difc.txt`, `wiki_uae_vat.txt`, `wiki_expo2020.txt`.
- Three fictional documents about **Falcon Dunes Trading LLC** -- a made-up UAE company. These are the interesting case: the base model cannot possibly know them, so if RAG answers questions about Falcon Dunes correctly, we know it is *actually* using our documents.


### 8.1 Step 1 -- Load the raw documents

In [ ]:
from pathlib import Path

DOCS_DIR = Path("/content/isa383/15_data")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

DOC_CONTENTS = {"falcondunes_leave_policy.txt":"FALCON DUNES TRADING LLC — INTERNAL LEAVE POLICY (FICTIONAL)\n\nNOTE: Falcon Dunes Trading LLC is a fictional company created for the ISA 383 Retrieval-Augmented Generation demonstration. Names, numbers, and policies in this document are invented and do not describe any real organisation.\n\nDocument ID: FD-HR-LEAVE-2026-01\nEffective date: 15 January 2026\nApproved by: Board of Directors, 10 January 2026\n\n1. ANNUAL LEAVE\n\n1.1 Full-time employees who have completed their probationary period (90 days) are entitled to 24 working days of paid annual leave per calendar year, exclusive of weekends and public holidays.\n\n1.2 Employees in their first year of service accrue annual leave on a pro-rata basis at a rate of 2 working days per completed month of service.\n\n1.3 Up to 10 days of unused annual leave may be carried forward to the next calendar year. Any balance beyond 10 days is forfeited on 31 December.\n\n1.4 Annual leave requests must be submitted via the Oasis HR portal at least 14 calendar days in advance and require approval from both the line manager and the head of department.\n\n2. SICK LEAVE\n\n2.1 Employees are entitled to 15 days of fully paid sick leave per calendar year, followed by 30 days at half pay, in accordance with UAE Labour Law.\n\n2.2 A medical certificate from an approved clinic is required for any continuous sick leave period exceeding 2 working days.\n\n3. PUBLIC HOLIDAYS\n\n3.1 Falcon Dunes observes all UAE federal public holidays as announced by the Ministry of Human Resources and Emiratisation.\n\n3.2 In addition, Falcon Dunes provides one discretionary \"Founders' Day\" holiday on the first working Thursday of November, commemorating the founding of the company in November 2011.\n\n4. COMPASSIONATE LEAVE\n\n4.1 Employees are entitled to 5 working days of compassionate leave on the death of an immediate family member (parent, spouse, child, sibling).\n\n4.2 Employees are entitled to 3 working days of compassionate leave on the death of a grandparent or parent-in-law.\n\n5. STUDY LEAVE\n\n5.1 Employees with more than 2 years of service may apply for up to 5 working days of paid study leave per calendar year to sit examinations relating to a professional qualification approved by the company (e.g., ACCA, CFA, PMP).\n\nQuestions regarding this policy should be directed to the Head of People Operations, Ms. Aisha Bin Harmal (a.binharmal@falcondunes.example).\n\n","falcondunes_org_chart.txt":"FALCON DUNES TRADING LLC — ORGANISATION CHART (FICTIONAL)\n\nNOTE: Falcon Dunes Trading LLC is a fictional company created for the ISA 383 Retrieval-Augmented Generation demonstration. All names and roles in this document are invented.\n\nDocument ID: FD-ORG-2026-Q1\nLast updated: 5 January 2026\nMaintained by: Office of the Chief Operating Officer\n\nOVERVIEW\n\nFalcon Dunes Trading LLC is headquartered in Business Bay, Dubai, with a secondary office in Jebel Ali Free Zone (warehouse and logistics) and a regional sales office in Riyadh. Total headcount as of Q1 2026 is 187 employees.\n\nEXECUTIVE LEADERSHIP\n\nChief Executive Officer (CEO): Mr. Khalid Al Marashi\n  - Joined the company as a co-founder in 2011.\n  - Reports to: Board of Directors.\n\nChief Financial Officer (CFO): Ms. Lina Abdelhamid\n  - Joined the company in 2019 from a regional audit firm.\n  - Reports to: CEO.\n\nChief Operating Officer (COO): Mr. Yusuf Haidari\n  - Joined the company in 2016.\n  - Reports to: CEO.\n\nChief Technology Officer (CTO): Dr. Priya Menon\n  - Joined the company in 2023.\n  - Reports to: CEO.\n\nDEPARTMENT HEADS\n\nHead of Sales (MEA region): Mr. Omar Daoudi\nHead of Sales (KSA): Ms. Noor Al Saeed (based in Riyadh)\nHead of Supply Chain: Mr. Rajiv Kulkarni (based in Jebel Ali)\nHead of Finance and Reporting: Mr. Adnan Qureshi\nHead of People Operations: Ms. Aisha Bin Harmal\nHead of IT and Digital: Mr. Tarek Al Hammadi\nHead of Compliance: Ms. Hanna Svensson\n\nBOARD OF DIRECTORS\n\nThe board consists of seven members, including three independent non-executive directors. The chairman is Mr. Abdulla Al Marashi (founder, non-executive).\n\nThe board meets four times per year (February, May, August, November). An audit and risk committee and a remuneration committee operate under the board, each chaired by an independent non-executive director.\n\n","falcondunes_q3_memo.txt":"FALCON DUNES TRADING LLC — INTERNAL Q3 PERFORMANCE MEMO (FICTIONAL)\n\nNOTE: Falcon Dunes Trading LLC is a fictional company created for the ISA 383 Retrieval-Augmented Generation demonstration. All figures are invented for teaching purposes.\n\nTo:       All Heads of Department\nFrom:     Lina Abdelhamid, Chief Financial Officer\nDate:     12 October 2025\nSubject:  Q3 2025 Performance Summary and Q4 Priorities\nDocument: FD-FIN-MEMO-2025-Q3\n\nREVENUE\n\nTotal Q3 2025 revenue was AED 142.6 million, up 11.4% year-on-year. Growth was driven primarily by the Saudi Arabia market, where revenue grew 28% year-on-year to AED 38.1 million, continuing the pattern established since the Riyadh office opened in 2023. UAE domestic revenue was flat at AED 91.8 million, with the shortfall against budget of AED 4.2 million attributable to delayed contract awards in the construction materials segment.\n\nGROSS MARGIN\n\nGross margin for Q3 was 22.8%, up 60 basis points on Q3 2024. The improvement reflects the full-quarter contribution from the new direct-sourcing agreement with Vietnamese suppliers signed in June.\n\nOPERATING EXPENSES\n\nOperating expenses were AED 21.3 million, in line with the Q3 budget. Notable items:\n\n  * Warehouse expansion at Jebel Ali (Phase 2) completed on 28 August, one week ahead of schedule and approximately AED 0.8 million under the original budget of AED 9.5 million.\n  * Oasis HR portal rollout completed company-wide on 15 September. Employee adoption in the first month was 93%, exceeding the 85% target.\n  * Salesforce CRM (\"Project Gazelle\") implementation continues. Go-live for the UAE sales team is scheduled for 1 December 2025; KSA migration will follow in Q1 2026.\n\nHEADCOUNT\n\nHeadcount at quarter-end was 182 (Q3 2024: 171). Net hiring in Q3 was +6. Notable senior hires: Ms. Hanna Svensson joined as Head of Compliance on 1 September.\n\nQ4 PRIORITIES\n\n  1. Successful Project Gazelle go-live for the UAE sales team on 1 December.\n  2. Close at least two of the three pending construction-materials contracts currently in final negotiation.\n  3. Finalise the 2026 budget and submit to the Board by 12 November.\n  4. Complete the annual external audit fieldwork (PwC) on schedule by mid-January 2026.\n\nQueries on any of the above should be directed to the CFO's office.\n\n","wiki_difc.txt":"Dubai International Financial Centre (DIFC)\n\nThe Dubai International Financial Centre (DIFC) is a special economic zone in Dubai covering 110 hectares (272 acres), established in 2004. DIFC is governed by an independent civil and commercial legal framework based on English common law. The zone has its own regulator, the Dubai Financial Services Authority (DFSA), and its own independent courts, the DIFC Courts.\n\nDIFC provides a platform for financial institutions and services companies, and is a leading financial hub for the Middle East, Africa, and South Asia (MEASA) region. As of 2024, more than 5,500 active registered companies operated within DIFC, including a large share of banking, capital markets, wealth and asset management, insurance, and professional services firms.\n\nKey regulatory and operational features of DIFC include a 0% tax rate on income and profits for a period of 50 years (with certain exceptions following the UAE's introduction of corporate tax in 2023), no restrictions on foreign ownership, no restrictions on capital repatriation, and the ability to employ foreign nationals. The zone uses the US dollar as its functional currency for most financial transactions.\n\nThe DIFC Courts operate in English and apply common law principles, making them particularly attractive to international businesses familiar with the English legal system. The courts have jurisdiction over civil and commercial disputes arising from within DIFC, and parties from outside DIFC can \"opt in\" to the jurisdiction by contract.\n\nDIFC is distinct from the Abu Dhabi Global Market (ADGM), which is a separate financial free zone established in 2013 on Al Maryah Island in Abu Dhabi.\n\n(Source: adapted from the Wikipedia article on Dubai International Financial Centre.)\n\n","wiki_expo2020.txt":"Expo 2020 Dubai\n\nExpo 2020 Dubai was a World Expo hosted by the United Arab Emirates from 1 October 2021 to 31 March 2022. The event was originally scheduled for October 2020 but was postponed by one year due to the COVID-19 pandemic. Despite the postponement, the name \"Expo 2020\" was retained for branding and contractual reasons.\n\nThe Expo took place on a 4.38 square-kilometre site in Dubai South, between Dubai and Abu Dhabi. It was the first World Expo to be held in the MEASA region. The theme was \"Connecting Minds, Creating the Future,\" supported by three sub-themes: Opportunity, Mobility, and Sustainability. The site contained three thematic districts organised around these sub-themes, each anchored by a landmark pavilion.\n\n192 countries participated, along with a number of multilateral organisations, corporations, educational institutions, and non-governmental organisations. Al Wasl Plaza, featuring the world's largest 360-degree projection surface, served as the centrepiece of the site.\n\nTotal visits during the six-month run exceeded 24 million, with participation from international visitors limited in the early months by ongoing travel restrictions from the pandemic.\n\nAfter the closing ceremony, the Expo site began its transformation into District 2020, a mixed-use community and business district intended to retain and repurpose approximately 80% of the Expo's built infrastructure. District 2020 hosts the offices of technology firms, startups, and educational institutions, and is positioned as a legacy project demonstrating the long-term economic impact of the Expo on Dubai.\n\n(Source: adapted from the Wikipedia article on Expo 2020.)\n\n","wiki_uae_vat.txt":"Value Added Tax (VAT) in the United Arab Emirates\n\nValue Added Tax (VAT) was introduced in the United Arab Emirates on 1 January 2018 at a standard rate of 5%. VAT is an indirect tax administered by the Federal Tax Authority (FTA) and applies to most goods and services supplied in the UAE, including imports.\n\nBusinesses with taxable supplies and imports exceeding AED 375,000 per year are required to register for VAT. Businesses with taxable supplies and imports exceeding AED 187,500 per year may register voluntarily. Once registered, a business must charge VAT on its taxable supplies, file periodic VAT returns (usually quarterly), and pay any net VAT due to the FTA.\n\nCertain supplies are zero-rated, meaning VAT is charged at 0% but the supplier can still reclaim input VAT. Zero-rated categories include exports of goods and services outside the GCC VAT implementing states, international transportation, the first supply of newly constructed residential buildings within three years of completion, certain education and healthcare services, and investment-grade precious metals.\n\nOther supplies are exempt from VAT, meaning no VAT is charged and input VAT cannot be reclaimed. Exempt categories include the supply of certain financial services, residential buildings (other than the first supply), bare land, and local passenger transport.\n\nVAT receipts are a significant source of non-oil revenue for the UAE federal budget. The introduction of VAT was part of a GCC-wide framework agreement, although not all GCC states have implemented VAT to date.\n\n(Source: adapted from the Wikipedia article on the Value Added Tax regime in the UAE.)\n\n"}

for filename, content in DOC_CONTENTS.items():
    (DOCS_DIR / filename).write_text(content, encoding="utf-8")

print(f"Prepared {len(DOC_CONTENTS)} course documents.")


### 8.2 Step 2 -- Chunk the text

We break each document into overlapping windows of a few hundred characters. Overlap matters: if a useful sentence straddles a chunk boundary, we still want it to appear in at least one whole chunk. A chunk of ~800 characters with 150-character overlap is a reasonable starting point for short business documents; you will tune this in the exercises.

In [19]:
def chunk_text(text, chunk_size=800, overlap=150):
    """Split text into overlapping character windows."""
    if len(text) <= chunk_size:
        return [text]
    out = []
    i = 0
    step = chunk_size - overlap
    while i < len(text):
        out.append(text[i : i + chunk_size])
        i += step
    return out


rows = []
for r in docs:
    for j, ch in enumerate(chunk_text(r["text"])):
        rows.append({"filename": r["filename"], "chunk_id": j, "chunk": ch})

chunks_df = pd.DataFrame(rows)
print("Total chunks:", len(chunks_df))
chunks_df.groupby("filename").size().rename("n_chunks")

Total chunks: 20


filename
falcondunes_leave_policy.txt    4
falcondunes_org_chart.txt       3
falcondunes_q3_memo.txt         4
wiki_difc.txt                   3
wiki_expo2020.txt               3
wiki_uae_vat.txt                3
Name: n_chunks, dtype: int64

### 8.3 Step 3 -- Embed the chunks

An **embedding** is a vector (a list of numbers, here 3072 of them) that captures the meaning of a piece of text. Two chunks about the same topic end up close together in this 3072-dimensional space; two unrelated chunks end up far apart. The trick is to compute the embeddings *once*, store them, and then do cheap vector arithmetic at query time.

Gemini's embedding model is `gemini-embedding-001` and uses the same API key. Usage remains subject to the current quota and pricing rules.

In [20]:
def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT", batch_size=100):
    """Embed a list of strings. Returns an (n, 3072) NumPy array."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        resp = client.models.embed_content(
            model="gemini-embedding-001",
            contents=batch,
            config=types.EmbedContentConfig(task_type=task_type),
        )
        vectors.extend([e.values for e in resp.embeddings])
    return np.asarray(vectors, dtype=np.float32)


chunk_vectors = embed_texts(chunks_df["chunk"].tolist(),
                            task_type="RETRIEVAL_DOCUMENT")

print("Vector matrix shape:", chunk_vectors.shape)
print("First embedding (first 5 values):", chunk_vectors[0, :5])

Vector matrix shape: (20, 3072)
First embedding (first 5 values): [-0.00180993  0.03972761  0.00140744 -0.06108786  0.00068552]


Note the `task_type` argument. Gemini returns slightly different embeddings depending on whether you tell it the text is a "document to be searched" (`RETRIEVAL_DOCUMENT`) or a "query that will search documents" (`RETRIEVAL_QUERY`). Matching them correctly on both sides improves retrieval quality.

### 8.4 Step 4 -- Store in a DataFrame

For a small corpus like this, a DataFrame + a NumPy matrix is plenty. For thousands of documents you would graduate to a dedicated **vector database** (ChromaDB, FAISS, Pinecone, etc.), but the interface and the mental model are identical.

In [21]:
chunks_df["embedding"] = list(chunk_vectors)
chunks_df.head(3)

,filename,chunk_id,chunk,embedding
0,falcondunes_leave_policy.txt,0,FALCON DUNES TRADING LLC — INTERNAL LEAVE POLI...,"[-0.001809928, 0.03972761, 0.0014074373, -0.06..."
1,falcondunes_leave_policy.txt,1,of service accrue annual leave on a pro-rata b...,"[0.0022123996, 0.029806858, -0.0076029995, -0...."
2,falcondunes_leave_policy.txt,2,clinic is required for any continuous sick le...,"[0.0061705187, 0.03148695, 0.00087824126, -0.0..."


### 8.5 Step 5 -- Cosine similarity search

Two vectors that point in similar directions represent similar meanings. The standard measure is **cosine similarity**: the dot product of the unit-normalised vectors. In NumPy it is one line of code.

In [22]:
def top_k_chunks(query, k=3):
    """Return the k chunks most similar to the query."""
    q_vec = embed_texts([query], task_type="RETRIEVAL_QUERY")[0]
    q_vec /= np.linalg.norm(q_vec)

    doc_vecs = chunk_vectors / np.linalg.norm(chunk_vectors, axis=1, keepdims=True)
    sims = doc_vecs @ q_vec

    top_idx = np.argsort(-sims)[:k]
    out = chunks_df.iloc[top_idx][["filename", "chunk_id", "chunk"]].copy()
    out["similarity"] = sims[top_idx]
    return out.reset_index(drop=True)


hits = top_k_chunks("How many days of annual leave do Falcon Dunes employees get?", k=3)
hits[["filename", "chunk_id", "similarity"]]

,filename,chunk_id,similarity
0,falcondunes_leave_policy.txt,1,0.823232
1,falcondunes_leave_policy.txt,2,0.793628
2,falcondunes_leave_policy.txt,0,0.745386


In [23]:
# Inspect the best-matching chunk
print(hits.iloc[0]["chunk"])

of service accrue annual leave on a pro-rata basis at a rate of 2 working days per completed month of service.

1.3 Up to 10 days of unused annual leave may be carried forward to the next calendar year. Any balance beyond 10 days is forfeited on 31 December.

1.4 Annual leave requests must be submitted via the Oasis HR portal at least 14 calendar days in advance and require approval from both the line manager and the head of department.

2. SICK LEAVE

2.1 Employees are entitled to 15 days of fully paid sick leave per calendar year, followed by 30 days at half pay, in accordance with UAE Labour Law.

2.2 A medical certificate from an approved clinic is required for any continuous sick leave period exceeding 2 working days.

3. PUBLIC HOLIDAYS

3.1 Falcon Dunes observes all UAE federal publ


In [24]:
print(hits.iloc[1]["chunk"])

 clinic is required for any continuous sick leave period exceeding 2 working days.

3. PUBLIC HOLIDAYS

3.1 Falcon Dunes observes all UAE federal public holidays as announced by the Ministry of Human Resources and Emiratisation.

3.2 In addition, Falcon Dunes provides one discretionary "Founders' Day" holiday on the first working Thursday of November, commemorating the founding of the company in November 2011.

4. COMPASSIONATE LEAVE

4.1 Employees are entitled to 5 working days of compassionate leave on the death of an immediate family member (parent, spouse, child, sibling).

4.2 Employees are entitled to 3 working days of compassionate leave on the death of a grandparent or parent-in-law.

5. STUDY LEAVE

5.1 Employees with more than 2 years of service may apply for up to 5 working days


The top match should be the Falcon Dunes leave-policy file -- exactly the document that contains the answer. Retrieval is working.

### 8.6 Step 6 -- Generate the final answer

Now we bundle the retrieved chunks into the prompt and ask Gemini to answer *using only that context*. The magic ingredient is a short system instruction that tells the model not to invent things outside the provided passages.

In [25]:
RAG_SYSTEM = textwrap.dedent("""
You are a careful assistant answering questions about a specific collection
of business documents. Use ONLY the context passages provided below. If the
context does not contain the answer, say "The documents do not say."
Quote the specific file where the answer comes from.
""").strip()


def rag_answer(question, k=3, *, model="gemini-2.5-flash", temperature=0.2):
    """Retrieve k relevant chunks, then ask Gemini using only that context."""
    hits = top_k_chunks(question, k=k)

    context_block = "\n\n".join(
        f"[Source: {row.filename}]\n{row.chunk}"
        for row in hits.itertuples()
    )

    prompt = f"""Context passages:
------
{context_block}
------

Question: {question}

Answer:"""

    return client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=RAG_SYSTEM,
            temperature=temperature,
            max_output_tokens=400,
        ),
    ).text

### 8.7 Test drive

Let us put the base model and the RAG model side by side on questions only the documents can answer.

In [26]:
def compare(q):
    print("Q:", q)
    print("\n[BASE MODEL]")
    print(ask(q, temperature=0.2))
    print("\n[RAG]")
    print(rag_answer(q))
    print("\n" + "-" * 70)


compare("How many days of paid annual leave do Falcon Dunes Trading employees get per year?")

Q: How many days of paid annual leave do Falcon Dunes Trading employees get per year?

[BASE MODEL]
Falcon Dunes Trading is a fictional company within the game Grand Theft Auto V.

As it's not a real-world company, there is no information available about the number of days of paid annual leave its employees would get. Fictional companies in games don't have real HR policies!

[RAG]
Full-time employees who have completed their probationary period are entitled to 24 working days of paid annual leave per calendar year.

Source: falcondunes_leave_policy.txt

----------------------------------------------------------------------


In [30]:
compare("Who is the CFO of Falcon Dunes Trading LLC, and when did she join?")

Q: Who is the CFO of Falcon Dunes Trading LLC, and when did she join?

[BASE MODEL]
I'm sorry, but I couldn't find any publicly available information about the CFO of Falcon Dunes Trading LLC, nor the date they joined.

This kind of information, especially for a private LLC, is often not disclosed publicly unless the company chooses to publish it on their own website or in specific business filings that are not easily searchable by the general public.

[RAG]
The CFO of Falcon Dunes Trading LLC is Ms. Lina Abdelhamid, and she joined the company in 2019.
Source: falcondunes_org_chart.txt

----------------------------------------------------------------------


In [27]:
compare("What was Falcon Dunes Trading's total revenue in Q3 2025, and where did the growth come from?")

Q: What was Falcon Dunes Trading's total revenue in Q3 2025, and where did the growth come from?

[BASE MODEL]
I cannot provide you with Falcon Dunes Trading's total revenue for Q3 2025 or details about its growth, for a few key reasons:

1.  **Future Date:** Q3 2025 has not occurred yet. Financial results are reported after the period has concluded.
2.  **Proprietary Information:** Even if it were a past quarter, specific revenue figures for a private company like Falcon Dunes Trading are generally not publicly disclosed. This information would be internal to the company.
3.  **Data Access:** My knowledge base does not include real-time or future financial data for specific companies.

To get this information, you would need to:
*   **Be an insider:** If you are part of Falcon Dunes Trading, you would have access to their internal financial reports.
*   **Wait for official reports:** If Falcon Dunes Trading were a publicly traded company (which it doesn't appear to be), they would rel

On every one of these questions the base model either refuses or invents something; the RAG version answers with the right number and cites the file. The whole pipeline is maybe fifty lines of Python, and it already beats a much larger, much more expensive fine-tune for this corpus.

### 8.8 It also handles questions about the real-world documents

In [32]:
compare("What is the standard VAT rate in the UAE, and when was VAT introduced?")

Q: What is the standard VAT rate in the UAE, and when was VAT introduced?

[BASE MODEL]
The standard VAT rate in the UAE is **5%**.

VAT was introduced in the UAE on **January 1, 2018**.

[RAG]
The standard VAT rate in the UAE is 5%, and VAT was introduced on 1 January 2018.
(Source: wiki_uae_vat.txt)

----------------------------------------------------------------------


In [33]:
compare("What was the theme of Expo 2020 Dubai?")

Q: What was the theme of Expo 2020 Dubai?

[BASE MODEL]
The theme of Expo 2020 Dubai was **"Connecting Minds, Creating the Future."**

This main theme was further explored through three sub-themes:
*   **Opportunity**
*   **Mobility**
*   **Sustainability**

[RAG]
The theme of Expo 2020 Dubai was "Connecting Minds, Creating the Future." (Source: wiki_expo2020.txt)

----------------------------------------------------------------------


## 9. When RAG Helps, and When It Does Not

| Situation                                          | RAG helps? |
|----------------------------------------------------|:----------:|
| Answering from a specific corpus (HR docs, contracts, product manuals) | Yes |
| Answering about recent events not in training data, when current sources are indexed | Yes |
| Answering common-knowledge questions (grammar, arithmetic, well-known history) | Not really |
| Creative writing / brainstorming                   | No         |
| Tasks that need reasoning over the entire corpus at once (e.g. "summarise everything") | Partly -- you may need a different pattern |

RAG is not a silver bullet. It is a *very* good pattern when your problem is "the model does not know this specific thing."

### The next level up

Once your corpus grows past a few hundred documents, upgrade:

- **Vector database.** `chromadb`, `faiss`, `qdrant`, `pinecone`. All of them expose more or less the same *embed-and-search* interface.
- **Better chunking.** Chunk on sentence or paragraph boundaries (`langchain-text-splitters`), tag chunks with metadata, drop tiny or duplicate chunks.
- **Re-ranking.** After the cosine-similarity step, run a small classifier that re-orders the top-k for relevance before passing to the LLM.
- **Frameworks.** `llama-index` and `langchain` are the two big ones; both are convenient, both are moving targets. Learn the plumbing first (this notebook), then pick a framework only when you outgrow the plumbing.

## 10. A Note on Academic Integrity

Using the Gemini API to *learn* faster (summarise a concept, debug an error, explain output) is encouraged. Using it to *submit* work that is not yours -- copying an LLM's solution into an assignment without understanding it -- is a violation of the course's AI-usage policy and the university's honour code.

A reliable test: if the model were turned off, could you still explain your submission line by line? If not, do not submit it.

## 11. Summary

1. **Every Generative AI call is still a request/response.** `client.models.generate_content(model=..., contents=...)` is the entire API.
2. **Customise with `GenerateContentConfig`.** System instructions set a persona; temperature controls creativity; max_output_tokens caps length.
3. **Chat state lives in a `Chat` object.** `client.chats.create(...)` gives you a conversational wrapper that remembers history.
4. **Base models do not know your private or recent data.** They will either refuse or hallucinate.
5. **RAG is the standard fix.** Embed your documents once, retrieve the top-k at query time, paste them into the prompt.
6. **Cosine similarity + NumPy is enough** for small corpora. Reach for a vector database only when you outgrow it.
7. **Embeddings and chat use the same Gemini API key.** Availability, quotas, and pricing depend on the student's current account tier.

In [ ]:
# @title Optionally replace the prepared corpus with your own text files
UPLOAD_OWN_DOCUMENTS = False  # @param {type:"boolean"}

if UPLOAD_OWN_DOCUMENTS:
    from google.colab import files

    uploaded = files.upload()
    text_files = {
        filename: content
        for filename, content in uploaded.items()
        if filename.lower().endswith(".txt")
    }
    if not text_files:
        raise ValueError("Choose at least one .txt file.")
    for existing_file in DOCS_DIR.glob("*.txt"):
        existing_file.unlink()
    for filename, content in text_files.items():
        (DOCS_DIR / filename).write_bytes(content)
    print(f"Uploaded {len(text_files)} text file(s). Rerun the corpus-loading cells above.")
else:
    print("Using the six prepared course documents.")


## 12. Exercises

1. **Persona.** Write a `system_instruction` that turns the bot into a tough but fair book reviewer. Have it review a book of your choice in five bullet points. Rerun twice with `temperature=0.2` and `temperature=0.9` and describe the difference.

2. **Multi-turn.** Build a 5-turn conversation with the tutor persona where each of your turns explicitly *references* what the bot said one turn earlier. Inspect `chat.get_history()` and describe what the object contains.

3. **Your own corpus.** Use the upload cell supplied with this exercise to choose three `.txt` files (articles, notes, one made-up company document). Re-run the chunking and embedding cells. Ask three questions and show the before/after of the base model vs RAG.

4. **Chunk size.** Re-run the chunking step with `chunk_size=200` and then with `chunk_size=1500`. For each setting, ask the same three questions. Which chunk size gives the most complete answers? Which gives the most *relevant* answers?

5. **Failure mode.** Ask your RAG system a question that the documents do *not* cover. Confirm that it answers "The documents do not say." rather than making something up.

6. **Optional -- model swap.** Re-run Question 4 with `model="gemini-2.5-pro"`. Do the answers improve? At what cost in latency?

---

*Submit your completed notebook by the deadline in the course schedule for Participation credit.*
